# 08 — Next Best Action Engine

## Objective
Turn the frozen CatBoost propensity model into a decision engine.

For each customer:
1. generate supported **product × channel** candidates;
2. score `P(conversion)` with the frozen CatBoost model;
3. estimate economic value:
   `expected_value = propensity × estimated_conversion_value - estimated_send_cost`;
4. rank actions and return one **Next Best Action**;
5. apply simple gates such as marketing consent and historical-support minimums.

### Important
- The model is **not retrained** here.
- The final test set is **not used to build the action catalog**.
- All monetary values remain in the customer's local country currency.
- Only Email, SMS, and Push are eligible because Voice/WhatsApp had no observed conversions in the historical dataset.


## 1. Install dependencies

In [1]:
!pip -q install catboost pyarrow scikit-learn joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.1 MB/s eta 0:00:00


## 2. Mount Drive and imports

In [2]:

from pathlib import Path
import sys, json, joblib
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

PROJECT_DIR = Path('/content/drive/MyDrive/03 Projects/Factored Hackathon')
SRC_DIR = PROJECT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from propensity.config import (
    TARGET,
    DATASET_PATH,
    ARTIFACT_DIR,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
)
from propensity.data import load_dataset, temporal_split
from propensity.features import INTERACTION_FEATURES
from propensity.nba_engine import (
    build_action_catalog,
    build_customer_snapshot,
    recommend_next_best_action,
)

FINAL_MODEL_DIR = Path(ARTIFACT_DIR) / 'final_evaluation' / 'catboost_20261004_222052'
MODEL_PATH = FINAL_MODEL_DIR / 'model.joblib'

NBA_DIR = Path(ARTIFACT_DIR) / 'nba'
NBA_DIR.mkdir(parents=True, exist_ok=True)

print('Model:', MODEL_PATH)
print('NBA artifacts:', NBA_DIR)
assert MODEL_PATH.exists(), f'Missing final frozen model: {MODEL_PATH}'


Mounted at /content/drive
Model: /content/drive/MyDrive/03 Projects/Factored Hackathon/artifacts/propensity/final_evaluation/catboost_20261004_222052/model.joblib
NBA artifacts: /content/drive/MyDrive/03 Projects/Factored Hackathon/artifacts/propensity/nba


## 3. Load the frozen model and pre-test history

In [3]:

model = joblib.load(MODEL_PATH)

df = load_dataset(DATASET_PATH)
split = temporal_split(df)

# Build the policy/catalog only from data that existed before the final holdout.
history_pretest = pd.concat(
    [split.train_df, split.validation_df],
    ignore_index=True,
)

print('Pre-test rows:', f'{len(history_pretest):,}')
print('Final test rows intentionally excluded from catalog:', f'{len(split.test_df):,}')


Pre-test rows: 1,031,925
Final test rows intentionally excluded from catalog: 182,105


## 4. Build the supported action catalog

In [4]:

action_catalog = build_action_catalog(history_pretest)

display(
    action_catalog[
        [
            'country',
            'promoted_product',
            'send_channel',
            'historical_sends',
            'historical_conversions',
            'historical_conversion_rate',
            'estimated_conversion_value',
            'estimated_send_cost',
        ]
    ].head(30)
)

print('Candidate action profiles:', len(action_catalog))
print('Countries:', action_catalog['country'].nunique())
print('Products:', action_catalog['promoted_product'].nunique())
print('Channels:', sorted(action_catalog['send_channel'].unique().tolist()))


,country,promoted_product,send_channel,historical_sends,historical_conversions,historical_conversion_rate,estimated_conversion_value,estimated_send_cost
0,Argentina,Cuenta Ahorro,Email,9220,68,0.007375,2604.650,0.00470
1,Argentina,Cuenta Ahorro,Push,4978,36,0.007232,1734.285,0.00050
2,Argentina,Cuenta Ahorro,SMS,10892,107,0.009824,2348.330,0.09050
3,Argentina,Cuenta Corriente,Email,16200,88,0.005432,2402.415,0.00470
4,Argentina,Cuenta Corriente,Push,2014,13,0.006455,2621.180,0.00040
5,Argentina,Cuenta Corriente,SMS,8860,70,0.007901,2673.305,0.09150
6,Argentina,Inversión,Email,8492,51,0.006006,2446.580,0.00470
7,Argentina,Inversión,Push,11032,81,0.007342,2785.300,0.00050
8,Argentina,Inversión,SMS,7632,76,0.009958,2398.435,0.09010
9,Argentina,Préstamo Hipotecario,Email,5169,33,0.006384,2662.210,0.00470


Candidate action profiles: 60
Countries: 3
Products: 7
Channels: ['Email', 'Push', 'SMS']


## 5. Build latest customer snapshots

In [5]:

customer_snapshot = build_customer_snapshot(history_pretest)

customer_id_col = next(
    c for c in ['customer_id', 'client_id', 'user_id']
    if c in customer_snapshot.columns
)

print('Customer id column:', customer_id_col)
print('Customers available for NBA:', f'{len(customer_snapshot):,}')
display(customer_snapshot[[customer_id_col, 'country', 'segment', 'accepts_marketing']].head())


Customer id column: customer_id
Customers available for NBA: 149,835


,customer_id,country,segment,accepts_marketing
0,CLI-P8F6JG7TN8YN,Argentina,Plus,False
1,CLI-T3NIVCSQQY7U,México,Plus,False
2,CLI-5YKKTQP6GVJS,Argentina,Basic,False
3,CLI-P21780PQ8D9W,Argentina,Plus,True
4,CLI-BMMOW2PQPKAW,México,Basic,False


## 6. Score a deterministic demo customer

In [6]:

MODEL_FEATURES_NBA = NUMERIC_FEATURES + CATEGORICAL_FEATURES + INTERACTION_FEATURES
CATEGORICAL_FEATURES_NBA = CATEGORICAL_FEATURES + INTERACTION_FEATURES

# Pick the first marketing-eligible customer deterministically.
eligible = customer_snapshot.copy()
if 'accepts_marketing' in eligible.columns:
    eligible = eligible[
        ~eligible['accepts_marketing'].astype(str).str.lower().isin(['false', '0', 'no', 'n'])
    ]

demo_customer = eligible.iloc[0]

recommendation, scored_actions = recommend_next_best_action(
    customer_row=demo_customer,
    action_catalog=action_catalog,
    model=model,
    model_features=MODEL_FEATURES_NBA,
    categorical_features=CATEGORICAL_FEATURES_NBA,
    min_historical_sends=100,
)

print('CUSTOMER:', recommendation.customer_id)
print('DECISION:', recommendation.decision)
print('PRODUCT:', recommendation.promoted_product)
print('CHANNEL:', recommendation.send_channel)
print('PROPENSITY:', recommendation.propensity_raw)
print('EXPECTED VALUE:', recommendation.expected_value)

display(
    scored_actions[
        [
            'rank',
            'promoted_product',
            'send_channel',
            'propensity_raw',
            'estimated_conversion_value',
            'estimated_send_cost',
            'expected_value',
            'historical_sends',
        ]
    ].head(15)
)


CUSTOMER: CLI-P21780PQ8D9W
DECISION: ACTION
PRODUCT: Tarjeta Crédito
CHANNEL: Push
PROPENSITY: 0.008405942144870617
EXPECTED VALUE: 26.788733259173966


,rank,promoted_product,send_channel,propensity_raw,estimated_conversion_value,estimated_send_cost,expected_value,historical_sends
0,1,Tarjeta Crédito,Push,0.008406,3186.940,0.0005,26.788733,5887
1,2,Préstamo Hipotecario,SMS,0.009133,2851.845,0.0923,25.953558,2325
2,3,Cuenta Corriente,SMS,0.009538,2673.305,0.0915,25.406779,8860
3,4,Inversión,Push,0.008484,2785.300,0.0005,23.630380,11032
4,5,Tarjeta Crédito,SMS,0.009169,2415.525,0.0911,22.056919,27590
5,6,Inversión,SMS,0.009193,2398.435,0.0901,21.958350,7632
6,7,Préstamo Hipotecario,Push,0.008321,2568.960,0.0005,21.375449,3676
7,8,Cuenta Corriente,Push,0.008082,2621.180,0.0004,21.185287,2014
8,9,Cuenta Ahorro,SMS,0.008880,2348.330,0.0905,20.762630,10892
9,10,Préstamo Personal,SMS,0.009561,2156.040,0.0921,20.521186,10412


## 7. Run a small multi-customer demo

In [7]:

demo_rows = []
for _, customer_row in eligible.head(25).iterrows():
    rec, _ = recommend_next_best_action(
        customer_row=customer_row,
        action_catalog=action_catalog,
        model=model,
        model_features=MODEL_FEATURES_NBA,
        categorical_features=CATEGORICAL_FEATURES_NBA,
        min_historical_sends=100,
    )
    demo_rows.append(rec.__dict__)

demo_results = pd.DataFrame(demo_rows)
display(demo_results)

print('\nDecision counts:')
print(demo_results['decision'].value_counts(dropna=False))

if (demo_results['decision'] == 'ACTION').any():
    print('\nRecommended channels:')
    print(
        demo_results.loc[demo_results['decision'] == 'ACTION', 'send_channel']
        .value_counts()
    )


,customer_id,decision,promoted_product,send_channel,propensity_raw,expected_conversion_value,estimated_send_cost,expected_value,historical_support
0,CLI-P21780PQ8D9W,ACTION,Tarjeta Crédito,Push,0.008406,3186.940,0.0005,26.788733,5887
1,CLI-WSYWH1GXLOQL,ACTION,Préstamo Hipotecario,SMS,0.008939,3120.640,0.0916,27.804556,3605
2,CLI-4GDVEM1L7VPY,ACTION,Préstamo Personal,SMS,0.009516,2597.110,0.0911,24.623379,26080
3,CLI-S7YM610MNEXR,ACTION,Cuenta Corriente,SMS,0.009432,2673.305,0.0915,25.123828,8860
4,CLI-LC3MDDQBCF0E,ACTION,Préstamo Personal,SMS,0.009386,2597.110,0.0911,24.284551,26080
5,CLI-U84E1HE0UALT,ACTION,Préstamo Personal,SMS,0.009257,2597.110,0.0911,23.951547,26080
6,CLI-GW978C7GS3Q6,ACTION,Préstamo Hipotecario,SMS,0.008842,3120.640,0.0916,27.501185,3605
7,CLI-O2CURN2552BE,ACTION,Préstamo Personal,SMS,0.009334,2597.110,0.0911,24.150412,26080
8,CLI-IEBLFOB3AT4Q,ACTION,Préstamo Personal,SMS,0.010332,2597.110,0.0911,26.743055,26080
9,CLI-P8A31F2HHMLJ,ACTION,Préstamo Personal,SMS,0.008853,2597.110,0.0911,22.901421,26080



Decision counts:
decision
ACTION    25
Name: count, dtype: int64

Recommended channels:
send_channel
SMS     20
Push     5
Name: count, dtype: int64


## 8. Sanity checks

In [8]:

assert set(action_catalog['send_channel'].unique()).issubset({'Email', 'SMS', 'Push'})
assert action_catalog['estimated_send_cost'].notna().all()
assert action_catalog['estimated_conversion_value'].notna().all()

if not scored_actions.empty:
    assert scored_actions['propensity_raw'].between(0, 1).all()
    assert np.isfinite(scored_actions['expected_value']).all()
    assert scored_actions['rank'].iloc[0] == 1
    assert scored_actions['expected_value'].is_monotonic_decreasing

print('Sanity checks passed.')
print('No final-test rows were used to build the action catalog.')


Sanity checks passed.
No final-test rows were used to build the action catalog.


## 9. Persist NBA artifacts

In [9]:

action_catalog_path = NBA_DIR / 'action_catalog_pretest.parquet'
snapshot_path = NBA_DIR / 'customer_snapshot_pretest.parquet'
demo_path = NBA_DIR / 'nba_demo_results.parquet'
metadata_path = NBA_DIR / 'nba_metadata.json'

action_catalog.to_parquet(action_catalog_path, index=False)
customer_snapshot.to_parquet(snapshot_path, index=False)
demo_results.to_parquet(demo_path, index=False)

metadata = {
    'model_source': str(MODEL_PATH),
    'model_family': 'CatBoost',
    'candidate_channels': ['Email', 'SMS', 'Push'],
    'model_features': MODEL_FEATURES_NBA,
    'categorical_features': CATEGORICAL_FEATURES_NBA,
    'min_historical_sends': 100,
    'economic_formula': 'propensity_raw * estimated_conversion_value - estimated_send_cost',
    'currency_note': 'Economic values are local currency within each customer country. Never aggregate across countries without FX conversion.',
    'catalog_data_scope': 'train + validation only; final test excluded',
    'next_step': 'Expose recommend_next_best_action as a deterministic tool for LangGraph.',
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

print('Saved:')
for p in [action_catalog_path, snapshot_path, demo_path, metadata_path]:
    print('-', p)


Saved:
- /content/drive/MyDrive/03 Projects/Factored Hackathon/artifacts/propensity/nba/action_catalog_pretest.parquet
- /content/drive/MyDrive/03 Projects/Factored Hackathon/artifacts/propensity/nba/customer_snapshot_pretest.parquet
- /content/drive/MyDrive/03 Projects/Factored Hackathon/artifacts/propensity/nba/nba_demo_results.parquet
- /content/drive/MyDrive/03 Projects/Factored Hackathon/artifacts/propensity/nba/nba_metadata.json


## Decision path after this notebook

If the demo produces sensible `ACTION` recommendations with:
- valid product/channel combinations,
- finite positive expected values,
- transparent propensity + economics,
- and consent/support gates,

then the decision engine is ready to become an application tool.

**Next notebook:** `09_langgraph_nba_agent.ipynb`

That notebook should wrap the deterministic NBA engine behind a tool such as:

```python
get_next_best_action(customer_id)
```

The LLM will explain/confirm/execute the recommendation, but it will **not invent the product, channel, probability, or expected value**.
